# Domain 2: Applications and Integration
**Exam**: Claude Certified Developer – Foundations (CCDV-F)
**Weighting**: 33.1% of exam
**Depth**: deep

## Domain Overview

This is the single largest domain on the exam (33.1%). It covers understanding
requirements, systems life cycle concepts, Claude API mechanics (messages, tools, streaming, vision,
thinking, caching, batch), software engineering foundations, Claude application design across
interfaces, and configuration management.

**Why it matters**: this domain is the core of "building production applications with Claude" —
everything from requirements to API mechanics to config management that a Claude application
depends on.

**Key concepts to know**: requirements analysis, SDLC integration, Claude API mechanics (especially
streaming, caching, and batch vs. realtime), software engineering foundations, cross-interface
application design, and configuration management (CLAUDE.md, settings.json, versioning).

In [ ]:
# Setup: a small helper so every cell in this notebook runs end-to-end,
# whether or not you have an ANTHROPIC_API_KEY configured.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    """Calls the real Claude API if ANTHROPIC_API_KEY is set; otherwise returns a
    simulated response so this notebook is runnable without API access."""
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception as e:
        return (f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY and `pip install anthropic` "
                f"to call the real API]\nPrompt received ({len(prompt)} chars): {prompt[:150]}...")

print("call_claude() ready. Set ANTHROPIC_API_KEY as an environment variable to hit the real API.")

## Key Concepts

### Understanding Requirements & Systems Life Cycle

Functional requirements describe what the system must do; infrastructure requirements
describe what it needs to run (compute, network, data access). Systems life cycle frameworks (plan,
develop, test, deploy, operate, retire) apply to Claude applications just as to any software system —
skipping stages (e.g., no operate/monitor plan) is a common early-stage mistake.

In [ ]:
def requirements_brief(functional: list[str], infrastructure: list[str]) -> dict:
    return {"functional": functional, "infrastructure": infrastructure}

print(requirements_brief(
    functional=["Summarize uploaded PDFs", "Answer follow-up questions about the summary"],
    infrastructure=["PDF text extraction service", "Claude API access", "Session storage"],
))

sdlc_stages = ["plan", "develop", "test", "deploy", "operate", "retire"]
print(sdlc_stages)

### Claude API Mechanics: messages, tools, streaming, vision, thinking, caching, batch

Core API mechanics: the Messages API structures conversations as role-tagged turns; tool
use lets Claude call functions; streaming delivers tokens incrementally for responsive UIs; vision
accepts images; extended thinking exposes reasoning; prompt caching reuses a stable prefix; the
Batch API trades latency for 50% cost savings on non-blocking workloads.

In [ ]:
def choose_api_mode(needs_realtime_ui: bool, is_blocking: bool, latency_tolerant_bulk: bool) -> str:
    if needs_realtime_ui:
        return "streaming Messages API"
    if latency_tolerant_bulk and not is_blocking:
        return "Message Batches API (50% cost savings, up to 24h window)"
    return "standard (non-streaming) Messages API"

print(choose_api_mode(needs_realtime_ui=True, is_blocking=True, latency_tolerant_bulk=False))
print(choose_api_mode(needs_realtime_ui=False, is_blocking=False, latency_tolerant_bulk=True))

### Software Engineering Foundations: REST, JSON, async, version control, SDLC integration

Claude applications are still normal software: they need REST API design discipline, valid
JSON handling (including malformed-output defensive parsing), asynchronous programming for
concurrent requests, and integration into the team's existing version-control and code-review
practices.

In [ ]:
import json

def safe_parse_json(text: str):
    try:
        return json.loads(text), None
    except json.JSONDecodeError as e:
        return None, f"malformed JSON: {e}"

print(safe_parse_json('{"summary": "ok"}'))
print(safe_parse_json('{summary: ok}'))  # malformed, defensive parsing needed

### Claude Application Design across interfaces (Claude Code, Desktop, claude.ai, API, SDKs)

Claude interprets instructions somewhat differently depending on interface: Claude Code
applies CLAUDE.md and tool permissions; the API applies only what's in the request (system prompt,
tools); claude.ai applies Project instructions. Application design must account for content
boundaries, schema design, and session hygiene (not letting stale context leak across unrelated
sessions) specific to the interface in use.

In [ ]:
interface_behavior = {
    "Claude Code": "applies CLAUDE.md hierarchy + tool permissions automatically",
    "Claude API (raw)": "only applies what's explicitly sent in the request - no implicit config",
    "claude.ai (Projects)": "applies Project instructions + knowledge sources",
}
for interface, behavior in interface_behavior.items():
    print(f"{interface:16s}: {behavior}")

### Configuration Management: CLAUDE.md, settings.json, model pinning, prompt versioning

Production Claude applications should pin model versions explicitly (avoid silently
picking up behavior changes on a new model release), version prompts alongside code (so prompt
changes are reviewable and revertible), and track plugin/dependency versions in configuration files
like `settings.json`.

In [ ]:
app_config = {
    "model_version": "claude-sonnet-4-5-20250929",  # pinned, not "latest"
    "prompt_version": "v3.2.0",
    "plugins": {"pdf-extractor": "1.4.0"},
}
print(app_config)
print("Risk of NOT pinning model_version: behavior can silently change on the next model release.")

## Mini Hands-On Project

**Covers**: all Domain 2 sub-objectives.

**Scenario**: Build the requirements, API strategy, and configuration for a PDF-summarization
application with both a real-time UI and a nightly bulk-reprocessing job.

### Step 1: Write the requirements brief

In [ ]:
print(requirements_brief(
    functional=["Summarize uploaded PDFs in real time", "Reprocess a backlog of 10,000 PDFs nightly"],
    infrastructure=["PDF parsing service", "Claude API access", "Object storage for PDFs"],
))

### Step 2: Choose the API mode for each part of the system

In [ ]:
print(choose_api_mode(needs_realtime_ui=True, is_blocking=True, latency_tolerant_bulk=False))
print(choose_api_mode(needs_realtime_ui=False, is_blocking=False, latency_tolerant_bulk=True))

### Step 3: Pin configuration and defensively parse output

In [ ]:
print(app_config)
print(safe_parse_json('{"summary": "PDF summary text"}'))

## Lab Exercises

### Exercise 1: Requirements gap analysis

**Objective**: Identify missing infrastructure requirements for a described application.

**Produce**: A list of missing infrastructure requirements.

In [ ]:
described_app = "A chatbot that answers questions using our internal wiki, with responses under 1 second."
stated_requirements = requirements_brief(
    functional=["Answer questions using the internal wiki"],
    infrastructure=["Claude API access"],
)
# TODO: identify at least 2 missing infrastructure requirements implied by "under 1 second" and "internal wiki"
missing = []  # TODO
print(missing)

# Expected output: Identifies missing items like a wiki retrieval/index service and a caching or streaming strategy to hit the latency target.

In [ ]:
# STRETCH GOAL: propose which SDLC stage (plan/develop/test/deploy/operate/retire) is most at risk
# of being skipped for a fast-moving internal tool like this, and why.
at_risk_stage = "..."  # TODO

# Expected output: A reasoned identification, likely 'operate' (monitoring/latency tracking) as commonly skipped.

### Exercise 2: API mode selection drill

**Objective**: Choose the right API mode for 4 new scenarios.

**Produce**: 4 mode choices with reasoning.

In [ ]:
scenarios = [
    ("Live typing indicator chatbot", True, True, False),
    ("Monthly analytics report over 50,000 records", False, False, True),
    ("Synchronous form-validation assistant", False, True, False),
    ("On-demand single document analysis, user waits for the result", False, True, False),
]
# TODO: call choose_api_mode for each
for desc, realtime, blocking, tolerant in scenarios:
    pass

# Expected output: Correct mode chosen per scenario, notably NOT recommending batch for anything blocking.

In [ ]:
# STRETCH GOAL: for the "on-demand single document analysis" case, decide if streaming would
# improve perceived latency even though it's not explicitly "realtime UI", and why.
streaming_decision = "..."  # TODO

# Expected output: A reasoned decision that streaming can improve perceived responsiveness even for a single-result wait.

### Exercise 3: Defensive JSON parsing pipeline

**Objective**: Extend safe_parse_json to retry with a cleanup step for common LLM JSON mistakes (e.g., trailing commas).

**Produce**: An updated parser handling at least one common malformed pattern.

In [ ]:
def safe_parse_json_v2(text: str):
    result, error = safe_parse_json(text)
    if error:
        # TODO: attempt one cleanup (e.g., strip trailing commas before closing braces/brackets) and retry
        cleaned = text  # TODO
        return safe_parse_json(cleaned)
    return result, error

print(safe_parse_json_v2('{"a": 1, "b": 2,}'))  # trailing comma - common LLM JSON mistake

# Expected output: Successfully parses the trailing-comma case after cleanup, where the original safe_parse_json would fail.

In [ ]:
# STRETCH GOAL: add a fallback that asks Claude to re-emit valid JSON if cleanup still fails
# (simulate with call_claude if you have SETUP_HELPER available, or describe the approach).
fallback_description = "..."  # TODO

# Expected output: A described (or implemented) fallback that asks the model to self-correct malformed JSON.

### Exercise 4: Cross-interface behavior comparison

**Objective**: Predict behavior differences for the same instruction across 3 interfaces.

**Produce**: 3 predictions with reasoning.

In [ ]:
instruction = "Always respond in formal English, never inject casual language."
interfaces_to_test = ["Claude Code (via CLAUDE.md)", "raw API call with no system prompt", "claude.ai Project with this in Project instructions"]
# TODO: for each interface, predict whether/how this instruction would actually be applied
for i in interfaces_to_test:
    pass

# Expected output: Correctly predicts the raw API call would NOT apply the instruction unless it's explicitly included in that request's system prompt.

In [ ]:
# STRETCH GOAL: write the minimal correct fix to ensure the raw API call also honors this instruction.
fix = "..."  # TODO
print(fix)

# Expected output: A fix stating the instruction must be included as a system parameter on every raw API request.

### Exercise 5: Configuration pinning audit

**Objective**: Audit an app_config for production-readiness gaps.

**Produce**: A list of configuration gaps and fixes.

In [ ]:
risky_config = {"model_version": "claude-latest", "prompt_version": None, "plugins": {"pdf-extractor": "latest"}}
# TODO: identify gaps (unpinned model, no prompt version, unpinned plugin) and propose fixes
gaps = []  # TODO
print(gaps)

# Expected output: Identifies all 3 unpinned/unversioned risks with concrete pinned-version fixes.

In [ ]:
# STRETCH GOAL: propose a process for safely updating model_version when a new model is released
# (e.g., staged rollout, eval suite gate) rather than pinning forever.
update_process = "..."  # TODO

# Expected output: A described staged-rollout or eval-gated process for safely updating pinned versions over time.

## Self-Assessment: Applications and Integration

Answer each question, then check the Answers cell below.


**Q1.** What is the difference between a functional requirement and an infrastructure requirement?

**Q2.** Which API approach should be used for a latency-tolerant, high-volume, non-blocking workload, and what is its main cost/latency tradeoff?

**Q3.** Why is defensive JSON parsing necessary even when using structured output techniques?

**Q4.** Why might the same system instruction behave differently across Claude Code, the raw API, and claude.ai?

**Q5.** Why should a production Claude application pin its model version rather than always using the latest?

## Answers


**A1.** Functional requirements describe what the system must do; infrastructure requirements describe what it needs to run (compute, data access, network).

**A2.** The Message Batches API - it offers up to 50% cost savings but has a processing window of up to 24 hours with no guaranteed latency SLA.

**A3.** Because even well-designed prompts/schemas can occasionally produce malformed output, and production systems must handle that gracefully rather than crashing.

**A4.** Because each interface applies a different implicit configuration layer (CLAUDE.md, nothing extra, or Project instructions) on top of the same underlying model.

**A5.** To avoid silently changing behavior when a new model version is released, which could break carefully-tuned prompts or evaluations without warning.

---
**Next**: [Domain 3: Claude Code](./03-claude-code.ipynb)
**Back**: [README](./README.md)